# testing stuff here

I'll be using this model: mshamrai/bert-base-ukr-eng-rus-uncased

Description: A multilingual BERT model fine-tuned to retain only Ukrainian, English, and Russian tokens.

Strengths: Compact size (~423 MB) and efficient for multilingual tasks.

Use Cases: Suitable for applications involving multiple languages, such as multilingual document retrieval.

Access: Available on Hugging Face: bert-base-ukr-eng-rus-uncased .
Hugging Face

In [1]:
model_name = "mshamrai/bert-base-ukr-eng-rus-uncased"

In [2]:
import pdfplumber

# If the PDF contains tables or more complicated layouts, pdfplumber can give better results than PyPDF2.
def document_to_text(source_srl: str) -> str:
    with pdfplumber.open(source_srl) as pdf:
        text = ""
        for page in pdf.pages:
            text += page.extract_text()
        return text

In [3]:
import re

# If the text has inconsistent formatting (and it does), regex will work better.
def split_into_paragraphs(text: str) -> list[str]:
    paragraphs = re.split(r'\n+', text.strip())
    return paragraphs

In [4]:
doc_url = "modeling_lab_doc.pdf"
lab_text = document_to_text(source_srl=doc_url)
paragraphs = split_into_paragraphs(text=lab_text)

In [5]:
paragraphs[100]

'відповідна величина береться зі знаком “+”. Якщо стрілка направлена від стану n –'

In [6]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def encode(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Convert to NumPy float32
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy().astype(np.float32).squeeze()

In [7]:
import numpy as np

vectors = np.array([encode(paragraph) for paragraph in paragraphs])
doc_ids = np.arange(len(vectors))

In [8]:
print("Vector shape:", vectors.shape)

Vector shape: (580, 768)


In [9]:
vectors[0].shape

(768,)

In [10]:
import hnswlib

dim = 768               # dimensionality of embeddings
num_elements = 10000    # max elements you expect to store

index = hnswlib.Index(space='cosine', dim=dim)
index.init_index(max_elements=num_elements, ef_construction=200, M=16)

In [11]:
print("Index dim:", index.dim)

Index dim: 768


In [ ]:
index.add_items(vectors, doc_ids)

: 